# Lab 09-01 -- Chunk embeddings + GMM clustering (RAPTOR step 1)

**Track 09 . RAPTOR** -- replacing flat chunk lists with a summary tree.

RAPTOR (Sarthi et al., 2024) builds a hierarchical index over a
corpus. The first half of that index is unsupervised: embed every
chunk and group the embeddings so that related chunks land in the
same cluster. Each cluster is small enough to be compressed by a
summary without losing the details -- the summarization happens in
lab 02, so this lab never calls an LLM.

```text
30 passages (rag-mini-wikipedia, deterministic head)
  -> BGE embeddings (BAAI/bge-base-en-v1.5, local, CPU)
  -> recursive GaussianMixture (2-component, split while > max_cluster_size)
  -> partition of clusters (every chunk appears exactly once)
  -> verification gate (--verify)
```

The clustering recipe from the paper is a recursive Gaussian
mixture model: fit a 2-component GaussianMixture over the current
chunk embeddings; if a resulting cluster is at most
max_cluster_size chunks, keep it as a leaf; otherwise split that
cluster again. The output is a partition: every chunk index appears
in exactly one cluster. That partition is the raw material for the
tree -- lab 02 turns each cluster into a summary node.


## Setup

This notebook mirrors `curriculum/09-raptor/01-chunk-clustering.py`
exactly -- the same verified code, split into cells. Unlike the
GraphRAG or agentic labs, there is **no Ollama prerequisite** for
this lab: no LLM is called at all. The only data prerequisite is
that the corpus exists on disk:
`Data/corpus/rag-mini-wikipedia/passages.parquet`, already fetched
by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it,
because a notebook has no `__file__` -- so every `Data/...` and
`tools/...` path resolves exactly like the lab script. From the
terminal the lab runs as:

```bash
python curriculum/09-raptor/01-chunk-clustering.py          # run + demo
python curriculum/09-raptor/01-chunk-clustering.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if
you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt -- the install
# is a no-op safety net for fresh environments):
#   langchain-huggingface -> the HuggingFace embedding backend behind embeddings/bge.py
#   pandas                -> read the rag-mini-wikipedia parquet
#   scikit-learn          -> GaussianMixture inside tools/raptor.py
%pip install langchain-huggingface pandas scikit-learn


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory -- this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) -- then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd  # noqa: E402

from embeddings.bge import BGEEmbedding  # noqa: E402
from tools.raptor import cluster_embeddings  # noqa: E402


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named
constant. `N_PASSAGES = 30` takes the **deterministic head** of
rag-mini-wikipedia -- no LLM calls means the number only controls
embedding and clustering time. `MAX_CLUSTER_SIZE = 10` caps how
many chunks a single cluster can hold before it gets split again;
anything bigger would lose too much detail in a single summary.
`SEED = 42` makes the GMM splits deterministic across runs. The
BGE embedder runs on CPU because Ollama typically holds the GPU's
VRAM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
N_PASSAGES = 30  # deterministic head; embedding + clustering only, no LLM calls
MAX_CLUSTER_SIZE = 10  # clusters larger than this get split recursively
SEED = 42  # deterministic GMM splits
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load -- first N passages of rag-mini-wikipedia

`passages.parquet` is a plain table with a `passage` column;
`head(n)` keeps the first `n` rows so every run works on the same
corpus slice. Each passage is loaded **whole**: there is no
chunking here, because the embedding and clustering lab consumes
the full text at once.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N passages of the rag-mini-wikipedia corpus
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


## 3. Experiment -- embed the chunks, then cluster them with the
recursive GMM

This is RAPTOR's unsupervised indexing step, in two halves:

- **Embed** -- every passage is turned into a 768-d BGE vector via
`BGEEmbedding` (local sentence-transformer, no API). The embedder
is instantiated once and called in bulk.
- **Cluster** -- `cluster_embeddings` (in `tools/raptor.py`) fits a
recursive Gaussian mixture: a 2-component GaussianMixture over the
current embeddings, then recurses into any cluster larger than
`MAX_CLUSTER_SIZE`. The result is a list of clusters, each a list
of chunk indices -- a partition covering every chunk exactly once.

The result is a single `exp` dict holding the passages, embeddings,
clusters, and timing -- the raw material lab 02 turns into a
summary tree.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed the chunks, then cluster them with the recursive GMM
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    t0 = time.perf_counter()
    embeddings = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    clusters = cluster_embeddings(
        embeddings, max_cluster_size=MAX_CLUSTER_SIZE, seed=SEED
    )
    cluster_s = time.perf_counter() - t0

    return {
        "passages": passages,
        "embeddings": embeddings,
        "clusters": clusters,
        "embed_s": embed_s,
        "cluster_s": cluster_s,
    }


## 4. Demo

The demo prints the artifact from two angles: the cluster sizes
(sorted largest first), and one sample passage per cluster (first
120 chars) so you can eyeball whether the grouping makes sense.
The takeaway is that clustering is the unsupervised half of
RAPTOR's index: each cluster groups chunks that a single summary
can compress, and the partition covers every chunk exactly once.
Lab 02 builds one summary node per cluster.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the partition
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-01 — Chunk embeddings + GMM clustering (RAPTOR step 1)")
    n = len(exp["passages"])
    clusters = exp["clusters"]
    print(f"{n} passages embedded in {exp['embed_s']:.1f}s; "
          f"{len(clusters)} clusters (cap {MAX_CLUSTER_SIZE}) found in "
          f"{exp['cluster_s']:.1f}s")
    print("=" * 66)

    sizes = sorted((len(c) for c in clusters), reverse=True)
    print(f"\n[1] Cluster sizes (largest first): {sizes}")

    print(f"\n[2] One sample passage per cluster (first 120 chars):")
    for i, cluster in enumerate(clusters):
        sample = exp["passages"][cluster[0]]
        print(f"    C{i} (size {len(cluster)}): {sample[:120]}")

    print(f"\n[3] Takeaway")
    print("    Clustering is the unsupervised half of RAPTOR's index: each")
    print("    cluster groups chunks that a single summary can compress.")
    print("    The partition covers every chunk exactly once, so lab 02 can")
    print("    build one summary node per cluster without losing chunks.")


## 5. Verification gate

The lab ships a `--verify` gate: four hard checks the clustering
must clear -- every chunk index assigned exactly once (exact-once
partition), cluster count in `[2, n//2]`, every cluster non-empty,
and every cluster size at most `MAX_CLUSTER_SIZE`. The gate turns
"the lab ran" into "the lab ran *correctly*" -- the same discipline
every lab in this repo applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    clusters = exp["clusters"]
    n = len(exp["passages"])
    flat = sorted(i for cluster in clusters for i in cluster)

    checks.append(("every chunk index assigned exactly once "
                   f"(union == set(range({n})))",
                   flat == list(range(n))))
    checks.append((f"cluster count in [2, {n // 2}] (got {len(clusters)})",
                   2 <= len(clusters) <= n // 2))
    checks.append(("every cluster is non-empty",
                   all(len(cluster) > 0 for cluster in clusters)))
    checks.append((f"every cluster size <= {MAX_CLUSTER_SIZE}",
                   all(len(cluster) <= MAX_CLUSTER_SIZE for cluster in clusters)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

No LLM calls means this runs fast: embedding 30 passages on CPU
takes a few seconds, and the recursive GMM clustering is nearly
instant. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo -- the artifact

Cluster sizes, one sample passage per cluster, and the takeaway --
the unsupervised half of the RAPTOR index.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS -- the same gate the CI-style
`--verify` run enforces. If any line shows FAIL, the clustering
pipeline misbehaved: check that `passages.parquet` is intact and
that `tools/raptor.py` is unchanged.


In [ ]:
verify_gate(exp)
